# Full Pipeline Diagnostic

**Before running:**
1. Settings → Accelerator → **T4 GPU**
2. Settings → Internet → **On**
3. Google Drive `musique_data/` must contain:
   - `musique_ans_v1.0_dev.jsonl`
   - `musique_ans_v1.0_train.jsonl`
   - `model1_complement.pt`
   - `model2_scorer.pt`

**What this checks:**
1. **Seed hit rate** — does BM25+dense find any gold passage in top-5 seeds?
2. **Graph edge coverage** — are the gold chain edges (A→B) in the graph?
3. **BFS upper-bound recall** — with perfect scoring, what is the max recall possible?
4. **Complement drift** — how similar is `complement(A,B)` to `m1_emb[B]`?

**Total runtime: ~4 hours on T4** (edge vector computation dominates — runs once, cached after)

In [ ]:
# ── [EDIT IF NEEDED] ───────────────────────────────────────────────────────────
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/retrieval"
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
# 1. Verify GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings → Accelerator → T4 GPU")
if torch.cuda.get_device_properties(0).major < 7:
    raise RuntimeError("GPU is P100 — change to T4 GPU")
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("CUDA OK")

In [ ]:
# 2. Clone repo + install deps
import os

repo_root = f"/kaggle/working/{REPO_NAME}"
if not os.path.isdir(repo_root):
    os.system(f"git clone {REPO_URL} {repo_root}")
else:
    os.system(f"cd {repo_root} && git pull")

os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())

os.system("pip install -q 'transformers>=4.35.0' 'rank_bm25>=0.2.2' 'sentence-transformers>=2.2.0' gdown")

try:
    import faiss
    print(f"faiss: OK (GPU={faiss.get_num_gpus()})")
except ImportError:
    os.system("pip install -q faiss-cpu")
    print("faiss-cpu installed")

print("Dependencies ready")

In [ ]:
# 3. Download data + checkpoints from Google Drive
import os, gdown

download_dir = "/kaggle/working/drive_data"
if not os.path.isdir(download_dir):
    print("Downloading from Google Drive...")
    gdown.download_folder(
        id=DRIVE_FOLDER_ID, output=download_dir,
        quiet=False, use_cookies=False,
    )
else:
    print("Drive data already downloaded")

print("\nFiles:")
for f in sorted(os.listdir(download_dir)):
    print(f"  {f:45s}  {os.path.getsize(f'{download_dir}/{f}')/1e6:7.1f} MB")

In [ ]:
# 4. Set up file paths
import os, shutil

drive = "/kaggle/working/drive_data"
os.makedirs(f"{WORK_DIR}/data/musique", exist_ok=True)
os.makedirs(f"{WORK_DIR}/models",       exist_ok=True)
os.makedirs(f"{WORK_DIR}/cache",        exist_ok=True)

for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src = f"{drive}/{fname}"
    dst = f"{WORK_DIR}/data/musique/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"  {fname}: OK")

for ckpt in ["model1_complement.pt", "model2_scorer.pt"]:
    src = f"{drive}/{ckpt}"
    dst = f"{WORK_DIR}/models/{ckpt}"
    if not os.path.exists(dst):
        shutil.copy(src, dst)
    print(f"  {ckpt}: OK ({os.path.getsize(dst)/1e6:.0f} MB)")

print("\nAll paths ready")

---
## Step 1 — Build Offline Caches
M1 embeddings (~6 min) → Graph (~2 min) → Edge vectors (~3.5 hours)  
All cached to disk — subsequent runs skip straight to diagnostics.

In [ ]:
# 5. Load data + models
import os, sys, pickle
import numpy as np
import torch
import faiss
from pathlib import Path
from transformers import BertTokenizerFast

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

from data_loader import load_musique
from baselines import BM25Retriever, DenseRetriever, reciprocal_rank_fusion
from graph_builder import build_graph
from model1_train import ComplementEncoder
from run_full_system import compute_m1_embeddings, compute_edge_vectors

CACHE_DIR  = Path(WORK_DIR) / "cache"
MODEL_DIR  = Path(WORK_DIR) / "models"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_NAME = "musique_val_None"
N_SEEDS    = 5

# Load data
print("Loading MuSiQue dev ...")
corpus, queries = load_musique(split="validation", cache=True)
id_to_idx = {c["chunk_id"]: i for i, c in enumerate(corpus)}
print(f"  {len(corpus):,} chunks | {len(queries):,} queries")

# Load Model 1
print("\nLoading Model 1 ...")
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
comp_enc  = ComplementEncoder().to(DEVICE)
comp_enc.load_state_dict(torch.load(f"{MODEL_DIR}/model1_complement.pt", map_location=DEVICE))
comp_enc.eval()
print("  Model 1 ready")

In [ ]:
# 6. Compute M1 embeddings (~6 min if not cached)
m1_emb = compute_m1_embeddings(
    comp_enc, corpus, tokenizer, DEVICE,
    cache_path=CACHE_DIR / f"m1_emb_{CACHE_NAME}.npy",
)
print(f"M1 embeddings: {m1_emb.shape}")

In [ ]:
# 7. Build graph with M1 embeddings (~2 min if not cached)
graph = build_graph(
    corpus,
    embeddings=m1_emb,
    cache_name=CACHE_NAME + "_m1",
)
graph_edges = set()
for a_id, neighbors in graph.items():
    for (b_id, _, _) in neighbors:
        graph_edges.add((a_id, b_id))
print(f"Graph: {len(graph):,} nodes | {len(graph_edges):,} directed edges")

In [ ]:
# 8. Compute edge vectors (~3.5 hours if not cached)
# This is the slow step — runs once and is cached.
edge_vecs = compute_edge_vectors(
    comp_enc, corpus, graph, tokenizer, DEVICE,
    cache_path=CACHE_DIR / f"edge_vecs_{CACHE_NAME}_m1.pkl",
)
print(f"Edge vectors: {len(edge_vecs):,}")

In [ ]:
# 9. Build seed retrievers (BM25 + dense, cached)
print("Building seed retrievers ...")
bm25  = BM25Retriever()
bm25.build(corpus,  cache_name=f"bm25_{CACHE_NAME}")
dense = DenseRetriever()
dense.build(corpus, cache_name=f"dense_{CACHE_NAME}")
print("Seed retrievers ready")

---
## Step 2 — Run Diagnostics

In [ ]:
# Helper
from collections import defaultdict
from tqdm.notebook import tqdm

def print_bar(label, value, width=30):
    filled = int(round(value * width))
    bar    = "█" * filled + "░" * (width - filled)
    print(f"  {label:45s} [{bar}]  {value*100:5.1f}%")

def cosine_sim(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na > 1e-9 and nb > 1e-9 else 0.0

print("Helpers ready")

In [ ]:
# DIAGNOSTIC 1 — SEED HIT RATE
print("=" * 70)
print("DIAGNOSTIC 1: SEED HIT RATE")
print("=" * 70)

seed_hit_any   = 0
seed_hit_all   = 0
seed_first_hop = 0
total          = 0
per_hop_seed   = defaultdict(lambda: {"hit": 0, "total": 0})

for q in tqdm(queries, desc="Seed check"):
    gold   = set(q["relevant_chunk_ids"])
    chain  = q.get("chain_chunk_ids", [])
    n_hops = q.get("hop_count", len(chain))

    d_list = dense.retrieve(q["question"], top_k=N_SEEDS * 3)
    b_list = bm25.retrieve(q["question"],  top_k=N_SEEDS * 3)
    seeds  = set(reciprocal_rank_fusion([d_list, b_list])[:N_SEEDS])

    seed_hit_any   += int(bool(gold & seeds))
    seed_hit_all   += int(gold.issubset(seeds))
    seed_first_hop += int(bool(chain) and chain[0] in seeds)
    total          += 1
    per_hop_seed[n_hops]["total"] += 1
    per_hop_seed[n_hops]["hit"]   += int(bool(gold & seeds))

print(f"\n  Total queries: {total:,}")
print_bar("Any gold passage in seeds",    seed_hit_any   / total)
print_bar("First chain passage in seeds", seed_first_hop / total)
print_bar("ALL gold passages in seeds",   seed_hit_all   / total)
print("\n  By hop count:")
for hop in sorted(per_hop_seed):
    d = per_hop_seed[hop]
    print_bar(f"  {hop}-hop: any gold in seeds", d["hit"] / max(d["total"], 1))

In [ ]:
# DIAGNOSTIC 2 — GRAPH EDGE COVERAGE
print("=" * 70)
print("DIAGNOSTIC 2: GRAPH EDGE COVERAGE")
print("=" * 70)

edge_total         = 0
edge_in_graph      = 0
full_chain_covered = 0
full_chain_total   = 0
per_hop_edge       = defaultdict(lambda: {"covered": 0, "total": 0})

for q in queries:
    chain  = q.get("chain_chunk_ids", [])
    n_hops = q.get("hop_count", len(chain))
    if len(chain) < 2:
        continue

    all_covered = True
    for i in range(len(chain) - 1):
        a_id, b_id = chain[i], chain[i + 1]
        in_graph   = (a_id, b_id) in graph_edges
        edge_total    += 1
        edge_in_graph += int(in_graph)
        if not in_graph:
            all_covered = False

    full_chain_covered += int(all_covered)
    full_chain_total   += 1
    per_hop_edge[n_hops]["total"]   += 1
    per_hop_edge[n_hops]["covered"] += int(all_covered)

print(f"\n  Total gold consecutive pairs: {edge_total:,}")
print_bar("Gold pair (A→B) has graph edge", edge_in_graph      / max(edge_total, 1))
print_bar("Full chain: ALL edges covered",  full_chain_covered / max(full_chain_total, 1))
print("\n  By hop count:")
for hop in sorted(per_hop_edge):
    d = per_hop_edge[hop]
    print_bar(f"  {hop}-hop: full chain in graph", d["covered"] / max(d["total"], 1))

In [ ]:
# DIAGNOSTIC 3 — BFS UPPER-BOUND RECALL
print("=" * 70)
print("DIAGNOSTIC 3: BFS UPPER-BOUND RECALL")
print("=" * 70)

per_hop_found = defaultdict(lambda: {"found": 0, "gold": 0, "queries": 0, "all_found": 0})

for q in tqdm(queries, desc="BFS check"):
    gold   = set(q["relevant_chunk_ids"])
    chain  = q.get("chain_chunk_ids", [])
    n_hops = q.get("hop_count", len(chain))

    d_list = dense.retrieve(q["question"], top_k=N_SEEDS * 3)
    b_list = bm25.retrieve(q["question"],  top_k=N_SEEDS * 3)
    seeds  = set(reciprocal_rank_fusion([d_list, b_list])[:N_SEEDS])

    reachable = set(seeds)
    frontier  = set(seeds)
    for _ in range(3):
        nxt = set()
        for node in frontier:
            for (nbr, _, _) in graph.get(node, []):
                if nbr not in reachable:
                    reachable.add(nbr)
                    nxt.add(nbr)
        frontier = nxt

    found_gold = gold & reachable
    per_hop_found[n_hops]["found"]     += len(found_gold)
    per_hop_found[n_hops]["gold"]      += len(gold)
    per_hop_found[n_hops]["queries"]   += 1
    per_hop_found[n_hops]["all_found"] += int(gold.issubset(reachable))

total_found = sum(d["found"] for d in per_hop_found.values())
total_gold  = sum(d["gold"]  for d in per_hop_found.values())

print(f"\n  BFS from seeds, depth 3, perfect scoring:")
print_bar("Overall passage recall upper bound", total_found / max(total_gold, 1))
print()
for hop in sorted(per_hop_found):
    d = per_hop_found[hop]
    print(f"  {hop}-hop  ({d['queries']:4d} queries):")
    print_bar(f"    passage recall upper bound", d["found"]     / max(d["gold"],    1))
    print_bar(f"    all gold reachable",         d["all_found"] / max(d["queries"], 1))

In [ ]:
# DIAGNOSTIC 4 — COMPLEMENT DRIFT
print("=" * 70)
print("DIAGNOSTIC 4: COMPLEMENT DRIFT")
print("=" * 70)

cosines = []
for q in queries:
    chain = q.get("chain_chunk_ids", [])
    for i in range(len(chain) - 1):
        a_id, b_id = chain[i], chain[i + 1]
        if (a_id, b_id) in edge_vecs and b_id in id_to_idx:
            cosines.append(cosine_sim(
                edge_vecs[(a_id, b_id)],
                m1_emb[id_to_idx[b_id]],
            ))
    if len(cosines) >= 2000:
        break

random_cosines = []
for (a_id, b_id), ev in edge_vecs.items():
    if b_id in id_to_idx:
        random_cosines.append(cosine_sim(ev, m1_emb[id_to_idx[b_id]]))
    if len(random_cosines) >= 500:
        break

mean_cos = float(np.mean(cosines))
print(f"\n  cosine(complement(A,B), m1_emb[B]) over {len(cosines)} gold edges:")
print(f"    mean: {mean_cos:.4f}  std: {np.std(cosines):.4f}  median: {np.median(cosines):.4f}")
print(f"\n  Random edges (500 samples):")
print(f"    mean: {np.mean(random_cosines):.4f}  std: {np.std(random_cosines):.4f}")
print()
for t in [0.95, 0.90, 0.80, 0.70]:
    pct = np.mean(np.array(cosines) > t)
    print(f"    gold edges with cosine > {t:.2f}: {pct*100:.1f}%")

print("\n  Interpretation:")
if mean_cos > 0.92:
    print("  HIGH DRIFT — complement ≈ passage embedding")
    print("  L_content dominates. Scoring with edge_vec ≈ scoring with m1_emb[B].")
    print("  Model 2 adds no signal beyond plain M1 cosine.")
elif mean_cos > 0.75:
    print("  MODERATE DRIFT — partial directional signal but L_content still dominates")
else:
    print("  LOW DRIFT — complement is distinct from passage embedding")

In [ ]:
# SUMMARY
print("=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"  1. Seeds catch any gold passage : {seed_hit_any/total*100:.1f}%")
print(f"  2. First chain passage in seeds : {seed_first_hop/total*100:.1f}%")
print(f"  3. Full gold chain in graph     : {full_chain_covered/max(full_chain_total,1)*100:.1f}%")
print(f"  4. BFS upper-bound recall       : {total_found/max(total_gold,1)*100:.1f}%")
print(f"  5. Complement-passage cosine    : {mean_cos:.3f}")
print()

bottlenecks = []
if seed_hit_any / total < 0.60:
    bottlenecks.append(f"SEED RETRIEVAL ({seed_hit_any/total*100:.0f}% hit rate) — seeds miss most gold passages")
if full_chain_covered / max(full_chain_total, 1) < 0.50:
    bottlenecks.append(f"GRAPH COVERAGE ({full_chain_covered/max(full_chain_total,1)*100:.0f}% full chains) — graph missing gold edges")
if total_found / max(total_gold, 1) < 0.70:
    bottlenecks.append(f"BFS CEILING ({total_found/max(total_gold,1)*100:.0f}%) — structural limit on max recall")
if mean_cos > 0.92:
    bottlenecks.append(f"COMPLEMENT DRIFT ({mean_cos:.3f}) — increase L_chain weight in Model 1")

if bottlenecks:
    print("  Bottlenecks found:")
    for b in bottlenecks:
        print(f"    * {b}")
else:
    print("  No single obvious bottleneck — check beam width / scoring margin")